In [1]:
# -------------------------------------------
# 
# generate and save climate input files 
# 
# -------------------------------------------

# Need the following 1d arrays --------------
# 
# [ time ]: decimal year 
# [ temperature ]: degrees C
# [ soil moisture ]: mm / m
# 
import os 

import icechunk 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.style as mplstyle
import numpy as np
import pandas as pd
import xarray as xr

import clim_process_helperfxns as cph

In [2]:
# select layers for mean column -------------
# for more info on soil layers, see: https://codes.ecmwf.int/grib/param-db/39 
soilwater_layers = ["volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2"]
soiltemperature_layers = ["soil_temperature_level_1", "soil_temperature_level_2"]
# select the 1d variables
runoff_vars = ["sub_surface_runoff"]

# select time resolutions to output ---------
default_resolution = True   # native time resolution
default_res_name = "hourly" # name for the native time resolution
monthly_mean = True         # average of each month (length = 12 * nyears)
monthly_ltm = True          # climatological average of each month (length = 12)
annual_mean = True          # average of each year (length = nyears)
annual_ltm = True           # climatological annual average (length = 1)

# do we need to convert era5 units ----------
era5_unit_conversions_on = True

# how many times to repeat the selected years in the climate file 
nyears_repeat = 1   # repeat timeseries for all resolutions
                    # (note, longterm mean resolutions are repeated nyears_data * nyears_repeat)
roundtime_to = 5    # number of decimal places to round decimal time (5 works for hourly)

# select start and end time + sites -----------
mintime = '2020-01-01'
maxtime = '2022-12-31'
site_dict = {
    "site_1": {'xlat': 38.148, 'xlon': 360 -121.449, 'name': 'central_valley'},    # central valley
    "site_2": {'xlat': 44.764, 'xlon': 360 -93.202, 'name': 'minneapolis'},     # minneapolis
    "site_3": {'xlat': 33.584, 'xlon': 360 -83.828, 'name': 'atlanta'},     # atlanta
    "site_4": {'xlat': 42.582, 'xlon': 360 -74.029, 'name': 'albany'},     # albany
}

# where to save the data -----------------------
# (note, we'll make a subdir that is `era5_mintime_maxtime`)
savepath = "/home/tykukla/ew-workflows/era5/scepter-ready"
save_maindir = os.path.join(savepath, f'era5_{mintime}_{maxtime}')
subdir_rule = "combined"

# input file for var details -------------------
inputvar_details_fn = "/home/tykukla/ew-workflows/era5/process/inputvar_details.txt"

# whether to save climate figure ---------------
save_clim_fig = True


In [3]:
# --- read in data
era5_dir = "era5/preprocessed_icechunk"
era5_bucket = "carbonplan-carbon-removal"

storage = icechunk.s3_storage(bucket=era5_bucket, prefix=era5_dir, from_env=True)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")
rtds = xr.open_zarr(session.store, consolidated=False)
rtds

<xarray.Dataset> Size: 354GB
Dimensions:                        (time: 184104, latitude: 105, longitude: 241)
Coordinates:
  * latitude                       (latitude) float32 420B 50.0 49.75 ... 24.0
    level                          int64 8B ...
  * longitude                      (longitude) float32 964B 235.0 ... 295.0
  * time                           (time) datetime64[ns] 1MB 2000-01-01 ... 2...
Data variables: (12/19)
    2m_temperature                 (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    evaporation                    (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    geopotential                   (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    land_sea_mask                  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    soil_temperature_level_2       (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    soil_temperature_level_3       (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    ...                             ...
    volumetric_soil_water_layer_2  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    total_precipitation            (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    surface_runoff                 (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_3  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_4  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_1  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
Attributes:
    last_updated:           2025-07-23 01:54:31.932125+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-07-17

## [1] Create climate var dataset 

In [4]:
# --- create climate var dataset 
dsvar, nyears_data = cph.create_climvars_ds(
    rtds,
    site_dict,
    mintime, 
    maxtime,
    soilwater_layers,
    soiltemperature_layers,
    runoff_vars,
    roundtime_to,
    era5_unit_conversions_on,
)

## [2] Create a dataset for each time resolution

In [5]:
ds_dict = cph.create_dsdict_across_timeResolutions(
    dsvar,
    default_res_name,
    default_resolution,
    monthly_mean,
    monthly_ltm,
    annual_mean,
    annual_ltm,
    roundtime_to,
)

## [3] Save all sites and time resolutions as SCEPTER input files

In [6]:
cph.save_all_case_climfiles_as_txt(
    ds_dict,
    nyears_data,
    save_maindir,
    inputvar_details_fn,
    save_clim_fig,
    subdir_rule,
)

Now saving: albany -- hourly
Now saving: atlanta -- hourly
Now saving: central_valley -- hourly
Now saving: minneapolis -- hourly
Now saving: albany -- monthly
Now saving: atlanta -- monthly
Now saving: central_valley -- monthly
Now saving: minneapolis -- monthly
Now saving: albany -- monthly_ltm
Now saving: atlanta -- monthly_ltm
Now saving: central_valley -- monthly_ltm
Now saving: minneapolis -- monthly_ltm
Now saving: albany -- yearly
Now saving: atlanta -- yearly
Now saving: central_valley -- yearly
Now saving: minneapolis -- yearly
Now saving: albany -- yearly_ltm
Now saving: atlanta -- yearly_ltm
Now saving: central_valley -- yearly_ltm
Now saving: minneapolis -- yearly_ltm


In [ ]:
# ----